# Lab 6 — Hadoop: FileSystem, MapReduce and YARN

### HDFS, MapReduce, YARN and how it all connects

So far we have used paths like `file:///tmp/...`, settings like `spark.hadoop.*`, and Spark has talked about "blocks" and "splits". All of this comes from **Hadoop**, the ecosystem Spark was built on.

In this lab we open up that layer: what Hadoop solved, why Spark partly replaced it, what is still used today, and how Spark and Pandas talk to it.

**Objectives**
- Understand the three pillars of Hadoop: HDFS, MapReduce and YARN
- Implement MapReduce in Python, Hadoop Streaming style, and compare it with Spark
- See how Spark uses the Hadoop FileSystem API (and why `s3a://` works)
- Start a single-node HDFS, use `hdfs dfs`, and read/write to it with Spark and Pandas
- Understand the role of YARN and the path from Hadoop to the cloud

---
## 1. Theory: where Hadoop came from

| Year | Milestone |
|-----|-------|
| 2003 | Google publishes the **GFS** (Google File System) paper |
| 2004 | Google publishes the **MapReduce** paper |
| 2006 | Doug Cutting and Mike Cafarella create **Hadoop**, an open source implementation of both ideas (Yahoo adopts it soon after) |
| 2012 | Hadoop 2 separates resource management into **YARN** |
| 2014 | **Spark** becomes a top-level Apache project: the same distributed model, but in memory |
| Today | In the cloud, HDFS gives way to object storage (S3, GCS, ADLS) and YARN to Kubernetes; on-premise, Hadoop remains in production at many companies |

The central idea of Hadoop: **instead of buying a giant server, use hundreds of ordinary machines** and accept that some will fail all the time. The software handles distribution and failures.

### The three pillars

```
  ┌────────────────────────────────────────────────────────────────────────┐
  │  PROCESSING        MapReduce   |   Spark, Hive, Flink...  (run here)   │
  ├────────────────────────────────────────────────────────────────────────┤
  │  RESOURCES         YARN: who runs what, on which machine, with how     │
  │                    much CPU and memory                                 │
  ├────────────────────────────────────────────────────────────────────────┤
  │  STORAGE           HDFS: huge files split into blocks and              │
  │                    replicated across machines                          │
  └────────────────────────────────────────────────────────────────────────┘
       machine 1        machine 2        machine 3   ...   machine N
```

| Pillar | Role | Most common modern equivalent |
|-------|-------|-------------------------------|
| **HDFS** | Distributed file system | S3, GCS, ADLS (object storage) |
| **YARN** | Cluster resource manager | Kubernetes, Databricks/EMR managers |
| **MapReduce** | Processing model and engine | Spark, Flink |

### Ecosystem

Dozens of projects grew up around these three pillars. Some you will still come across:

| Project | What it is | Status today |
|---------|---------|---------------|
| **Hive** | SQL over files on HDFS | The **Hive Metastore** is still used as a table catalog by Spark, Trino, etc. |
| **HBase** | NoSQL database on HDFS | Used in low-latency niches |
| **Sqoop** | Import from relational databases | Retired (Apache Attic) |
| **Oozie** | Job orchestration | Replaced by Airflow and similar tools |
| **Parquet / ORC** | Columnar formats | **More widely used than ever** |

---
## 2. Setup

This notebook runs in any environment with PySpark. Notebook 02 installs Hadoop and starts a real HDFS (tested for Colab/Linux).

In [1]:
# !pip install -q pyspark==4.0.1 pyarrow

In [2]:
import os, re, subprocess, shutil, time, random, glob
from collections import defaultdict
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Lab 6 - Hadoop")
    .master("local[*]")
    # spark.hadoop.* goes straight to the Hadoop configuration. Here: 4 MB blocks on the local disk
    # (the default is 32 MB), so we can see several blocks with a small file (section 3)
    .config("spark.hadoop.fs.local.block.size", str(4 * 2**20))
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext

BASE = "/tmp/lab6"
os.makedirs(BASE, exist_ok=True)

def sh(cmd, check=True):
    """Runs a shell command and shows its output (equivalent to ! in a notebook)."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}")

print("Spark            :", spark.version)
print("Hadoop (client)  :", spark._jvm.org.apache.hadoop.util.VersionInfo.getVersion())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/04 18:28:00 WARN Utils: Your hostname, ERJ1752, resolves to a loopback address: 127.0.1.1; using 10.23.20.50 instead (on interface enp0s31f6)
26/11/04 18:28:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/04 18:28:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark            : 4.0.1
Hadoop (client)  : 3.4.1


Look at the second line: **PySpark already ships with the Hadoop client libraries** inside it. Every `spark.read` and `df.write` goes through them.

---
## 3. HDFS: the distributed file system

### Blocks and replication

HDFS splits each file into **large blocks** (128 MB by default) and stores each block on **3 different machines**.

```
  file sales.csv (300 MB)  ->  block A (128 MB) | block B (128 MB) | block C (44 MB)

               NameNode (metadata: "sales.csv = A, B, C; A is on 1, 2 and 4...")
                  │
      ┌───────────┼─────────────┬──────────────┬─────────────┐
   DataNode 1   DataNode 2   DataNode 3    DataNode 4    DataNode 5
   [A] [C]      [A] [B]      [B] [C]       [A] [C]       [B]

   If DataNode 2 dies, A and B remain available on other machines,
   and the NameNode orders the missing copies to be recreated.
```

| Component | Function | Stores |
|------------|--------|--------|
| **NameNode** | The file system's "index" | Directory tree and where each block is, **in memory** |
| **DataNode** | Stores the blocks | Blocks as regular files on the local disk |
| **Client** | Asks the NameNode where the blocks are and reads directly from the DataNodes | — |

### Why such large blocks

- Fewer blocks means less metadata in the NameNode's memory.
- Sequentially reading 128 MB amortizes the cost of seeking the block.
- Each block is a **unit of parallelism**: one task processes one block.

### Limitations (and why they still matter)

| Limitation | Consequence |
|-----------|--------------|
| Files are **write-once** (append only) | No UPDATE, hence the need for Delta/Iceberg (Lab 8) |
| The NameNode keeps everything in memory (~150 bytes per file/block) | The **small files problem**: millions of KB-sized files bring down the NameNode |
| Storage and processing on the same machines | To get more disk you buy more CPU with it (and vice versa) |

### Data locality: "move the code to the data"

Hadoop's most important principle. Transferring 1 TB over the network is slow; transferring a 10 KB program is instant. So the scheduler tries to run each task **on the machine that already has the block**.

```
   Without locality                        With locality
   DataNode 1 ──[block A]──net───▶ Task    DataNode 1 [block A] ◀── Task runs here
```

Spark inherited this: in the Stages tab of the Spark UI, the **Locality Level** column shows `NODE_LOCAL`, `RACK_LOCAL` or `ANY`.

### 3.1 The Hadoop FileSystem API

Spark never talks directly to the disk, to HDFS or to S3. It talks to **Hadoop's `FileSystem` interface**, and the **URL scheme** picks the implementation.

```
   spark.read.parquet("<scheme>://path")
                  │
     Hadoop FileSystem API (same interface for all)
                  │
     ┌────────────┼──────────────┬─────────────────┬────────────────┐
   file://      hdfs://        s3a://            gs://            abfs://
   local disk   HDFS           Amazon S3         Google Cloud     Azure Data Lake
                               (hadoop-aws)      (gcs-connector)  (hadoop-azure)
```

We can access this API directly through PySpark's Java gateway:

In [3]:
jvm  = spark._jvm
conf = spark._jsc.hadoopConfiguration()

for scheme in ["file", "hdfs", "webhdfs", "s3a", "gs", "abfs"]:
    try:
        impl = jvm.org.apache.hadoop.fs.FileSystem.getFileSystemClass(scheme, conf).getName()
        print(f"{scheme + '://':<11} -> {impl}")
    except Exception:
        print(f"{scheme + '://':<11} -> (connector not installed)")

file://     -> org.apache.hadoop.hive.ql.io.ProxyLocalFileSystem
hdfs://     -> org.apache.hadoop.hdfs.DistributedFileSystem
webhdfs://  -> org.apache.hadoop.hdfs.web.WebHdfsFileSystem
s3a://      -> (connector not installed)
gs://       -> (connector not installed)
abfs://     -> (connector not installed)


`hdfs://` already works because the HDFS client ships with PySpark. For `s3a://`, `gs://` and `abfs://` you need to add the matching connector (for example, the `hadoop-aws` package for S3). **This is how Spark reads from S3 to this day: through the Hadoop project's S3A connector.**

Let's create a text file and inspect it with the same API Spark uses internally.

In [4]:
VOCAB = ("data spark hadoop hdfs yarn mapreduce cluster block file table python pandas "
         "parquet delta streaming window model pipeline shuffle partition memory disk network "
         "driver executor task stage job lake bronze silver gold").split()
weights = [1 / (i + 1) for i in range(len(VOCAB))]          # Zipf distribution: a few very frequent words

rng = random.Random(42)
CORPUS = f"{BASE}/corpus.txt"
with open(CORPUS, "w") as f:
    for _ in range(600_000):
        f.write(" ".join(rng.choices(VOCAB, weights, k=12)) + "\n")

path   = jvm.org.apache.hadoop.fs.Path(f"file://{CORPUS}")
fs     = path.getFileSystem(conf)
status = fs.getFileStatus(path)

print(f"Size             : {status.getLen() / 1e6:.1f} MB")
print(f"Block size       : {status.getBlockSize() / 2**20:.0f} MB  (set in the setup)")
print(f"Replication      : {status.getReplication()}")
print(f"Logical blocks   : {-(-status.getLen() // status.getBlockSize())}")

Size             : 45.0 MB
Block size       : 4 MB  (set in the setup)
Replication      : 1
Logical blocks   : 11


### 3.2 Blocks and Spark partitions

In the old RDD-based API (`sc.textFile`), Spark uses Hadoop's `InputFormat`, and **each block becomes a partition**. In the DataFrame API (`spark.read`), Spark computes the splits on its own (`spark.sql.files.maxPartitionBytes`, Lab 4) and uses the blocks to decide **where** to run each task.

With the 4 MB blocks set in the setup:

In [5]:
size_mb = os.path.getsize(CORPUS) / 2**20
print(f"{size_mb:.0f} MB file with 4 MB blocks -> {-(-size_mb // 4):.0f} blocks")
print("sc.textFile  (Hadoop InputFormat)   :", sc.textFile(CORPUS).getNumPartitions(), "partitions")
print("spark.read   (DataFrame)            :", spark.read.text(CORPUS).rdd.getNumPartitions(), "partitions")

43 MB file with 4 MB blocks -> 11 blocks
sc.textFile  (Hadoop InputFormat)   : 11 partitions
spark.read   (DataFrame)            : 11 partitions


Remember the **`spark.hadoop.`** prefix used in the setup: any Hadoop setting can be passed to Spark this way. That is how S3 access is configured (`spark.hadoop.fs.s3a.access.key`, `spark.hadoop.fs.s3a.endpoint`...).

---
## 4. MapReduce

### The model

All processing is expressed in two functions you write, plus a step the framework does:

```
  INPUT            MAP                 SHUFFLE & SORT              REDUCE            OUTPUT
  (blocks)   (per line: emits       (framework: groups       (per key: combines
              key/value pairs)       all the values           the values)
                                     of each key)

  "spark hdfs"  -> (spark,1)(hdfs,1)                     hdfs  -> [1,1]  -> (hdfs, 2)
  "hdfs yarn"   -> (hdfs,1)(yarn,1)   ─ network/disk ──▶ spark -> [1,1]  -> (spark, 2)
  "spark"       -> (spark,1)                             yarn  -> [1]    -> (yarn, 1)
```

| Step | Who writes it | Parallelism |
|-------|-------------|-------------|
| Map | You | One task per input block |
| Shuffle & Sort | The framework | Sends each key to a reducer (`hash(key) % R`) |
| Reduce | You | R tasks, each receives a set of **sorted** keys |

This is exactly what happens in a Spark `groupBy`: the map is the stage before the `Exchange` and the reduce is the stage after it (Lab 4).

### 4.1 Step-by-step simulation

Before running it for real, a pure Python simulation with 3 lines, 2 mappers and 2 reducers, to see each step:

In [6]:
lines = ["spark hdfs spark", "hdfs yarn", "spark yarn yarn"]
splits = [lines[:2], lines[2:]]                 # 2 "blocks" -> 2 mappers
N_REDUCERS = 2

def map_fn(line):
    for word in line.split():
        yield word, 1

def reduce_fn(word, counts):
    return word, sum(counts)

# MAP: each mapper processes its own block
map_out = [[kv for line in split for kv in map_fn(line)] for split in splits]
for i, out in enumerate(map_out):
    print(f"mapper {i}:", out)

# SHUFFLE: each pair goes to reducer hash(key) % R; then sort and group by key
reducer_in = [defaultdict(list) for _ in range(N_REDUCERS)]
for out in map_out:
    for word, n in out:
        reducer_in[sum(map(ord, word)) % N_REDUCERS][word].append(n)   # simple, deterministic hash

print()
for r, groups in enumerate(reducer_in):
    print(f"reducer {r} receives:", dict(sorted(groups.items())))

# REDUCE
print()
for r, groups in enumerate(reducer_in):
    print(f"reducer {r} emits   :", [reduce_fn(w, c) for w, c in sorted(groups.items())])

mapper 0: [('spark', 1), ('hdfs', 1), ('spark', 1), ('hdfs', 1), ('yarn', 1)]
mapper 1: [('spark', 1), ('yarn', 1), ('yarn', 1)]

reducer 0 receives: {'yarn': [1, 1, 1]}
reducer 1 receives: {'hdfs': [1, 1], 'spark': [1, 1, 1]}

reducer 0 emits   : [('yarn', 3)]
reducer 1 emits   : [('hdfs', 2), ('spark', 3)]


### 4.2 Hadoop Streaming: MapReduce with Python scripts

Hadoop is written in Java, but **Hadoop Streaming** lets you use any program that reads from standard input and writes to standard output. The contract is simple:

- The **mapper** reads lines and prints `key<TAB>value`.
- The framework **sorts** everything by key.
- The **reducer** reads the already sorted lines, so all occurrences of a key arrive together.

This means you can test a whole MapReduce job in the terminal, without a cluster:

```
cat input | python mapper.py | sort | python reducer.py
            └── map ───────┘  └shuffle┘ └── reduce ───┘
```

In [7]:
MAPPER = r'''
import sys, re
for line in sys.stdin:
    for word in re.findall(r"[a-z]+", line.lower()):
        print(f"{word}\t1")
'''

REDUCER = r'''
import sys
current, count = None, 0
for line in sys.stdin:
    word, n = line.rstrip("\n").split("\t")
    if word != current:              # new key: the previous one is done (the input is sorted)
        if current is not None:
            print(f"{current}\t{count}")
        current, count = word, 0
    count += int(n)
if current is not None:
    print(f"{current}\t{count}")
'''

with open(f"{BASE}/mapper.py", "w") as f:  f.write(MAPPER)
with open(f"{BASE}/reducer.py", "w") as f: f.write(REDUCER)

print("Mapper output (start):")
sh(f"head -n 1 {CORPUS} | python3 {BASE}/mapper.py")

Mapper output (start):
block	1
data	1
spark	1
data	1
python	1
file	1
memory	1
data	1
hadoop	1
data	1
data	1
hdfs	1


In [8]:
t0 = time.perf_counter()
sh(f"cat {CORPUS} | python3 {BASE}/mapper.py | LC_ALL=C sort | python3 {BASE}/reducer.py > {BASE}/wordcount_mr.tsv")
t_mr = time.perf_counter() - t0

print(f"MapReduce (local pipeline): {t_mr:.1f} s\n")
mr = pd.read_csv(f"{BASE}/wordcount_mr.tsv", sep="\t", names=["word", "count"])
mr.sort_values("count", ascending=False).head(8)

MapReduce (local pipeline): 4.4 s



,word,count
3,data,1774344
25,spark,888427
10,hadoop,591002
11,hdfs,442645
31,yarn,355290
14,mapreduce,295570
2,cluster,253342
0,block,222112


> The local pipeline is slow because it runs everything in **one process per step**, and `sort` has to read all of the mapper's output before releasing the reducer. In a Hadoop cluster, hundreds of mappers and reducers run in parallel. Even so, each job writes its output to HDFS, and that is costly (section 4.4).

### 4.3 The same word count in Spark

Three versions: RDD (the original API, which mimics map/reduce), DataFrame and SQL.

In [9]:
t0 = time.perf_counter()
rdd_counts = (
    sc.textFile(CORPUS)
    .flatMap(lambda line: re.findall(r"[a-z]+", line.lower()))   # map: line -> words
    .map(lambda w: (w, 1))                                       # map: word -> (word, 1)
    .reduceByKey(lambda a, b: a + b)                             # shuffle + reduce
    .collect()
)
t_rdd = time.perf_counter() - t0

t0 = time.perf_counter()
df_counts = (
    spark.read.text(CORPUS)
    .select(F.explode(F.split(F.lower("value"), " ")).alias("word"))
    .groupBy("word").count()
    .toPandas()
)
t_df = time.perf_counter() - t0

print(f"Spark RDD       : {t_rdd:.1f} s")
print(f"Spark DataFrame : {t_df:.1f} s")
print(f"MapReduce local : {t_mr:.1f} s")

check = mr.merge(df_counts, on="word", suffixes=("_mr", "_spark"))
print("\nSame result:", (check["count_mr"] == check["count_spark"]).all() and len(check) == len(mr))

Spark RDD       : 1.5 s
Spark DataFrame : 1.3 s
MapReduce local : 4.4 s

Same result: True


### 4.4 Why Spark replaced MapReduce

Word count is a single step. Real algorithms (ETL with several joins, iterative machine learning, graphs) are **chains of steps**. In MapReduce, each step is a separate job that **reads from HDFS and writes to HDFS**:

```
  MapReduce
  HDFS ─▶ [map ─ shuffle ─ reduce] ─▶ HDFS ─▶ [map ─ shuffle ─ reduce] ─▶ HDFS ─▶ ...
          job 1 (disk, 3 replicas)           job 2 (disk, 3 replicas)

  Spark
  HDFS ─▶ [stage 1] ─shuffle─▶ [stage 2] ─shuffle─▶ [stage 3] ─▶ HDFS
          a single job, optimized plan, intermediate data in memory when possible
```

| | MapReduce | Spark |
|-|-----------|-------|
| Between steps | Writes to HDFS (disk + replication) | Memory / local disk |
| Iterative algorithms (ML) | One job per iteration | Data cached between iterations (Lab 5) |
| API | map and reduce, nothing else | DataFrames, SQL, streaming, ML |
| Optimizer | None | Catalyst + AQE |
| Languages | Java (Python via Streaming) | Python, Scala, Java, R, SQL |
| Minimum job latency | Tens of seconds | Sub-second |

Spark kept the good ideas (splitting into partitions, moving code to the data, tolerating failures by recomputing) and removed the disk bottleneck.

---
## 5. YARN: who decides where everything runs

In a shared cluster, several teams run jobs at the same time. YARN distributes CPU and memory among them.

```
                    ResourceManager (1 per cluster)
                    "I have 400 cores and 2 TB free"
                         │
      ┌──────────────────┼──────────────────┐
  NodeManager 1     NodeManager 2      NodeManager 3     (1 per machine)
  ┌───────────┐     ┌───────────┐      ┌───────────┐
  │ container │     │ container │      │ container │
  │ AppMaster │     │ executor  │      │ executor  │
  │ (driver)  │     │ executor  │      │           │
  └───────────┘     └───────────┘      └───────────┘
```

| YARN | Spark |
|------|-------|
| ResourceManager | Equivalent to the cluster "master" |
| ApplicationMaster | Negotiates resources for **one** application; in `cluster` mode, the driver runs inside it |
| Container | A slice of CPU + memory; each Spark **executor** is a container |
| Queues | Limit how much each team can use |

On the Spark side, the only change is `--master`. The same code we ran with `local[*]` goes to the cluster like this:

```bash
spark-submit \
  --master yarn \
  --deploy-mode cluster \            # driver inside the cluster (client = driver on your machine)
  --num-executors 10 \
  --executor-cores 4 \
  --executor-memory 8g \
  --queue analytics \
  pipeline_silver.py
```

| `--master` | Where it runs |
|------------|-----------|
| `local[*]` | Your machine (every lab so far) |
| `yarn` | Hadoop cluster |
| `k8s://https://...` | Kubernetes |
| `spark://host:7077` | Spark's own standalone cluster |

---
## 6. Hadoop today: what stayed and what changed

### Separating storage and processing

The biggest change of the last 10 years. In classic Hadoop, the same machines store the data and process it. In the cloud, storage is a separate, cheap service, and processing spins up and down on demand.

```
  Classic Hadoop (on-premise)                  Cloud lakehouse
  ┌──────────┐ ┌──────────┐ ┌──────────┐       ┌────────┐ ┌────────┐   Spark clusters
  │ CPU+RAM  │ │ CPU+RAM  │ │ CPU+RAM  │       │ Spark  │ │ Spark  │   spin up and down
  │ HDFS     │ │ HDFS     │ │ HDFS     │       │ ETL    │ │ ML     │   per job
  └──────────┘ └──────────┘ └──────────┘       └───┬────┘ └───┬────┘
   cluster on 24h, sized for                        └────┬─────┘
   peak disk AND peak CPU                       ┌────────▼────────────────────┐
                                                │ S3 / GCS / ADLS             │
                                                │ Parquet + Delta / Iceberg   │
                                                └─────────────────────────────┘
```

| Component | Status |
|------------|----------|
| **HDFS** | Still strong on-premise; replaced by object storage in the cloud |
| **MapReduce** | Practically only in legacy systems |
| **YARN** | Still widely used (including on AWS EMR); Kubernetes is growing fast |
| **FileSystem API** | **Ubiquitous**: it is how Spark, Hive, Trino and Flink access S3/GCS/ADLS |
| **Hive Metastore** | Still a common table catalog, alongside Unity Catalog, Glue, Polaris |
| **Parquet/ORC** | The standard for analytical storage |
| **Ideas** (blocks, locality, fault tolerance through recomputation) | Everywhere, including tools like Polars and DuckDB |

### Why Hadoop is still worth learning

- Many large companies (banks, telecom, government, retail) still run on-premise Hadoop clusters, and migrations take years.
- Spark errors in production often come from the Hadoop layer: `FileNotFoundException`, output commits, `s3a` configuration, HDFS permissions.
- Understanding blocks, replicas and locality explains a lot about Spark performance.

---
## 7. Exercises

In [10]:
# ── Exercise 1 ──────────────────────────────────────────────────────────────
# MapReduce with scripts: compute total revenue (price * quantity) by region from the CSV below,
# writing a mapper and a reducer in Hadoop Streaming style and running them with
#   cat | mapper | sort | reducer
# Check the result with Spark or Pandas.
# Hint: the mapper should skip the header and emit "region<TAB>revenue".

ORDERS_CSV = f"{BASE}/orders.csv"
rng = random.Random(7)
with open(ORDERS_CSV, "w") as f:
    f.write("order_id,region,price,quantity\n")
    for i in range(200_000):
        f.write(f"{i},{rng.choice(['North','Northeast','Midwest','Southeast','South'])},"
                f"{rng.uniform(10, 500):.2f},{rng.randint(1, 5)}\n")

# Your solution:
MAPPER_REV = r'''
import sys
next(sys.stdin)                                   # skips the header
for line in sys.stdin:
    _, region, price, qty = line.rstrip("\n").split(",")
    print(f"{region}\t{float(price) * int(qty)}")
'''
REDUCER_REV = r'''
import sys
current, total = None, 0.0
for line in sys.stdin:
    region, value = line.rstrip("\n").split("\t")
    if region != current:
        if current is not None:
            print(f"{current}\t{total:.2f}")
        current, total = region, 0.0
    total += float(value)
if current is not None:
    print(f"{current}\t{total:.2f}")
'''
with open(f"{BASE}/mapper_rev.py", "w") as f:  f.write(MAPPER_REV)
with open(f"{BASE}/reducer_rev.py", "w") as f: f.write(REDUCER_REV)

sh(f"cat {ORDERS_CSV} | python3 {BASE}/mapper_rev.py | LC_ALL=C sort | python3 {BASE}/reducer_rev.py > {BASE}/rev_mr.tsv")
mr_rev = pd.read_csv(f"{BASE}/rev_mr.tsv", sep="\t", names=["region", "revenue"])

sp_rev = (spark.read.csv(ORDERS_CSV, header=True, inferSchema=True)
          .groupBy("region").agg(F.round(F.sum(F.col("price") * F.col("quantity")), 2).alias("revenue"))
          .toPandas())

check = mr_rev.merge(sp_rev, on="region", suffixes=("_mr", "_spark"))
check["diff"] = (check["revenue_mr"] - check["revenue_spark"]).abs().round(2)
check

,region,revenue_mr,revenue_spark,diff
0,Midwest,30548956.32,30548956.32,0.0
1,North,30319378.35,30319378.35,0.0
2,Northeast,30795282.83,30795282.83,0.0
3,South,30445992.80,30445992.80,0.0
4,Southeast,30566306.95,30566306.95,0.0


In [11]:
# ── Exercise 2 ──────────────────────────────────────────────────────────────
# HDFS sizing. Use 128 MB blocks, replication 3 and ~150 bytes of memory
# on the NameNode per object (each file and each block is an object).
#   a) How many blocks and how much raw disk space do 10 TB of data take?
#   b) How much NameNode memory do those 10 TB consume if stored in 1 GB files?
#   c) What if the same 10 TB are in 100 KB files?
# Relate answer (c) to the small files problem from Labs 4 and 8.

# Your solution:
TB, GB, KB, MB = 2**40, 2**30, 2**10, 2**20
BLOCK, REPL, BYTES_PER_OBJ = 128 * MB, 3, 150
data = 10 * TB

blocks = data // BLOCK
print(f"a) {blocks:,} blocks | raw space = {data * REPL / TB:.0f} TB")

def namenode_memory(file_size):
    n_files  = data // file_size
    n_blocks = n_files * max(1, -(-file_size // BLOCK))      # every file takes at least 1 block
    return n_files, (n_files + n_blocks) * BYTES_PER_OBJ

mems = {}
for label, size in [("b) 1 GB files  ", GB), ("c) 100 KB files", 100 * KB)]:
    n, mems[label] = namenode_memory(size)
    print(f"{label}: {n:>12,} files -> {mems[label] / GB:8.2f} GB of memory on the NameNode")
print(f"\nSame data volume, {max(mems.values()) / min(mems.values()):,.0f}x more memory.")

# Same volume, much more memory. That is why we compact small files
# (coalesce before writing, Lab 4; OPTIMIZE in Delta, Lab 8). S3 has no NameNode,
# but the cost shows up another way: millions of list and read requests.

a) 81,920 blocks | raw space = 30 TB
b) 1 GB files  :       10,240 files ->     0.01 GB of memory on the NameNode
c) 100 KB files:  107,374,182 files ->    30.00 GB of memory on the NameNode

Same data volume, 2,330x more memory.


**Exercise 3 (discussion).** A company has an on-premise Hadoop cluster of 40 machines, with 2 PB on HDFS, Hive jobs and some old MapReduce ones. It wants to migrate to the cloud.

1. Which parts of the stack need to change and which can stay almost the same?
2. Do the existing Spark jobs need to be rewritten? What changes in them?
3. What cost problem does separating storage and processing solve?

**Solution:**

1. **Changes:** storage (HDFS to S3/GCS/ADLS), resource management (YARN to EMR/Dataproc/Databricks/Kubernetes) and the catalog (Hive Metastore to Glue/Unity Catalog, or a managed Metastore). **Stays almost the same:** formats (Parquet/ORC), Spark code and Hive SQL (which runs on Spark SQL with few adjustments). The MapReduce jobs are the natural candidates to rewrite in Spark.
2. Generally no. What changes are the **paths** (`hdfs://` to `s3a://` or `gs://`), the **settings** (credentials, connectors, output committers) and `--master`. The DataFrame logic stays the same, thanks to the Hadoop FileSystem API.
3. In classic Hadoop, getting more disk means buying more machines (with CPU and RAM included), and the cluster stays on 24 hours, sized for the peak. In the cloud, you pay for cheap storage all the time and for compute **only while jobs run**, with a different size for each job.

In [12]:
spark.stop()